# PortfolioAgent - Week 17 API Notebook
**Applied GenAI & Agentic AI Engineering Course - Week 17 - AgentForge**

An A2A-compatible agent that answers questions about the AgentForge capstone. The model calls tools over the capstone's `AGENTS.md`, week detail files, architecture and eval results, and every fact in an answer cites the source a tool returned. This notebook walks every API endpoint - the build cards, the agent's trace, a follow-up, the approval gate, A2A and the guards - mostly two ways: **cURL (Windows cmd)** and **Python (`requests`)**. The browser UI (`/`) and the rendered README (`/readme`) are pages, so open those in a browser.

---
### Before you start
1. Server running: `uvicorn app.main:app --reload --reload-dir app --port 8000`
2. `.env` filled in with `OPENAI_API_KEY` (see `.env.example`)
3. Run the **Setup** cell below once. The cells that ask the agent call the real model - each question is two or three model calls.

> **Windows note:** curl cells use `%%cmd` with `\"` to escape inner quotes.
>
> **Against your Render URL:** set `BASE` to it and set the `ADMIN_TOKEN` environment variable first - the `[admin]` routes (`/actions`, `/approve`, `/audit`, `/memory`) need it there. Locally, with no token set, every route is open.

In [ ]:
# Setup - run this cell first
import os, requests, json

BASE = 'http://localhost:8000'
TOKEN = os.environ.get('ADMIN_TOKEN')                       # only needed against a deployed URL
ADMIN = {'Authorization': f'Bearer {TOKEN}'} if TOKEN else {}
print('Setup complete. BASE:', BASE, '| admin token:', 'set' if TOKEN else 'not set (local)')

---
## 1 - Health - `GET /health`
Both model tiers, both caps, and the price table.

In [ ]:
%%cmd
curl -s http://localhost:8000/health

In [ ]:
r = requests.get(f'{BASE}/health')
print('Status:', r.status_code)
print(json.dumps(r.json(), indent=2))

---
## 2 - Build cards - `GET /portfolio`
What the browser UI opens on: one card per week, parsed from the **Build history** in `data/AGENTS.md`, plus both eval gates' pass rates. Each card opens a read page, `GET /portfolio/{week}`. Text parsing only: no model call, no cost.

In [ ]:
pf = requests.get(f'{BASE}/portfolio').json()
print(pf['title'], '-', len(pf['builds']), 'builds | gates:', pf['gates'])
for b in pf['builds']:
    print(f"W{b['week']:<3}{b['name']:28s}{BASE}/portfolio/{b['week']}")

---
## 3 - A2A Agent Card - `GET /.well-known/agent-card.json`
The discovery document any A2A client reads first. It advertises two skills (`capstone_qa`, `request_intro`) and, under `supportedInterfaces`, **where and how to call the agent**: `/a2a`, JSON-RPC, protocol 1.0.

In [ ]:
%%cmd
curl -s http://localhost:8000/.well-known/agent-card.json

In [ ]:
card = requests.get(f'{BASE}/.well-known/agent-card.json').json()
print('Agent:', card['name'])
print('Skills:', [s['id'] for s in card['skills']])
print(json.dumps(card, indent=2))

---
## 4 - A2A - talk to the agent the way another agent would
JSON-RPC `SendMessage` to the URL the card advertises. The reply is a `Message` from `ROLE_AGENT`: one text part, plus `metadata` with the tier, model, cost, the tools the agent called and the citations it verified - the same loop as `POST /ask`.

In [ ]:
%%cmd
curl -s -X POST http://localhost:8000/a2a -H "Content-Type: application/json" -H "A2A-Version: 1.0" -d "{\"jsonrpc\": \"2.0\", \"id\": 1, \"method\": \"SendMessage\", \"params\": {\"message\": {\"messageId\": \"m-1\", \"role\": \"ROLE_USER\", \"parts\": [{\"text\": \"Tell me about your AgentForge\"}]}}}"

In [ ]:
card = requests.get(f'{BASE}/.well-known/agent-card.json').json()
a2a_url = card['supportedInterfaces'][0]['url']             # discovered, not hard-coded
url = a2a_url.replace('http://localhost:8000', BASE)        # same path on any BASE
reply = requests.post(url, headers={'A2A-Version': '1.0'}, json={
    'jsonrpc': '2.0', 'id': 1, 'method': 'SendMessage',
    'params': {'message': {'messageId': 'm-1', 'role': 'ROLE_USER',
                           'parts': [{'text': 'Tell me about your AgentForge'}]}}}).json()
msg = reply['result']['message']
print('role:', msg['role'], '| tier:', msg['metadata']['tier'])
print('tools:', msg['metadata']['toolsCalled'])
print('cited:', msg['metadata']['citations'])
print(msg['parts'][0]['text'])

---
## 5 - A2A - the other core methods answer, with the spec's errors
This agent replies with a `Message` and never creates a task, so the task methods answer the way A2A 1.0 says an agent like this must: an empty `ListTasks` page, `TaskNotFound` (-32001), `UnsupportedOperation` for streaming (-32004). A client asking for another protocol version gets `VersionNotSupported` (-32009) - and so does one that sends no `A2A-Version` header, because the spec reads a missing header as 0.3.

In [ ]:
def rpc(method, params=None, version='1.0'):
    body = {'jsonrpc': '2.0', 'id': 7, 'method': method, 'params': params or {}}
    return requests.post(f'{BASE}/a2a', json=body, headers={'A2A-Version': version}).json()

print('ListTasks           :', rpc('ListTasks', {'pageSize': 10})['result'])
for method in ('GetTask', 'SendStreamingMessage', 'GetExtendedAgentCard'):
    err = rpc(method, {'id': 't-1'})['error']
    print(f'{method:20s}:', err['code'], err['message'])
err = rpc('SendMessage', {'message': {'messageId': 'm', 'role': 'ROLE_USER',
                                      'parts': [{'text': 'What is the cost ceiling?'}]}}, version='0.3')['error']
print('A2A-Version 0.3     :', err['code'], err['message'])
err = requests.post(f'{BASE}/a2a', json={'jsonrpc': '2.0', 'id': 8, 'method': 'ListTasks'}).json()['error']
print('no A2A-Version      :', err['code'], err['message'])

---
## 6 - Ask a lookup - the SMALL tier, and the tools the model chose
`What did you build in Week 4?` has no reasoning cue, so the classifier routes it to the small tier (gpt-5.4-nano). The model then **chooses** its tools by function calling - here `get_build(4)` - and must cite what they returned. `grounded` is true only if every citation matches a source a tool returned in this run.

In [ ]:
%%cmd
curl -s -X POST http://localhost:8000/ask -H "Content-Type: application/json" -d "{\"question\": \"What did you build in Week 4?\"}"

In [ ]:
d = requests.post(f'{BASE}/ask', json={'question': 'What did you build in Week 4?'}).json()
print('tier :', d['tier'], '| model:', d['model'], '| model calls:', d['model_calls'])
print('trace:', ' -> '.join(f"{s['tool']}({', '.join(map(str, s['args'].values()))})" for s in d['tools_called']))
print('cited:', d['citations'], '| invented:', d['unverified_citations'], '| grounded:', d['grounded'])
print('cost : $%.6f  (saved $%.6f vs the frontier)' % (d['cost_usd'], d['saved_usd']))
print(d['answer'])

---
## 7 - What the agent retrieved
Each step of the trace carries the text that tool call pulled - the context the model answered from. The UI shows the same thing under **Retrieved context**.

In [ ]:
for s in d['tools_called']:
    used = 'cited' if s['source'] in d['citations'] else 'not cited'
    print(f"--- {s['tool']}({s['args']}) -> {s['source']} [{used}]")
    print((s['content'] or s['error'] or '')[:400])

---
## 8 - A reasoning question - the FRONTIER tier, several tools
A `why` / `compare` question escalates to gpt-5.4-mini. Comparing two weeks, the model typically calls a tool per week - and may go deeper with `get_week_details` or `get_architecture(week=...)`.

In [ ]:
d = requests.post(f'{BASE}/ask', json={'question': 'Compare what Week 5 and Week 6 built.'}).json()
print('tier :', d['tier'], '| model:', d['model'], '| model calls:', d['model_calls'])
print('trace:', [f"{s['tool']}({s['args']})" for s in d['tools_called']])
print('cited:', d['citations'])
print(d['answer'])

---
## 9 - A follow-up - the last turns travel with the question
The browser keeps the last 3 turns and sends them as `history`; the server stores none of it. The tier is decided on the NEW question only.

In [ ]:
first = requests.post(f'{BASE}/ask', json={'question': 'What did you build in Week 16?'}).json()
history = [{'question': 'What did you build in Week 16?', 'answer': first['answer']}]
d = requests.post(f'{BASE}/ask', json={'question': 'Why did you choose that design?',
                                      'history': history}).json()
print('trace:', [f"{s['tool']}({s['args']})" for s in d['tools_called']])
print(d['answer'])

---
## 10 - The approval gate - an intro request waits for the student
Ask to get in touch and the model calls `request_intro`. It creates an action at `input-required` and does **nothing else**; only the student (the admin token on a deploy) decides it - from the inbox, `GET /actions`, then `POST /approve`. Approval is state the system owns: no sentence from the visitor or the model can approve it.

In [ ]:
d = requests.post(f'{BASE}/ask', json={'question': "I'm hiring an AI engineer at Acme - please "
                                      "ask the student to contact me at jane@example.com."}).json()
pending = d['pending_action']
print('agent  :', d['answer'])
print('status :', pending and pending['status'], '| id:', pending and pending['id'])
print('inbox  :', [a['id'] for a in requests.get(f'{BASE}/actions', headers=ADMIN).json()])
done = requests.post(f'{BASE}/approve', json={'action_id': pending['id'], 'approve': True}, headers=ADMIN).json()
print('after approve:', done['status'], '-', done['result'])

---
## 11 - Failure - too-short question, or too much history (422)
`min_length=8`, and at most 3 history turns of 2,000 characters each; both rejected before any model call.

In [ ]:
print('hi        :', requests.post(f'{BASE}/ask', json={'question': 'hi'}).status_code, '(expected 422)')
turn = {'question': 'q', 'answer': 'a'}
r = requests.post(f'{BASE}/ask', json={'question': 'What did you build in Week 4?', 'history': [turn] * 4})
print('4 turns   :', r.status_code, '(expected 422)')

---
## 12 - Failure - off-topic question (no tool, out of scope)
Not about the capstone: the model calls no tool and replies with the `OUT_OF_SCOPE` sentinel, so `grounded` is false.

In [ ]:
d = requests.post(f'{BASE}/ask', json={'question': 'What is the capital of France?'}).json()
print('tools   :', d['tools_called'], '| grounded:', d['grounded'])
print('answer  :', d['answer'])

---
## 13 - Failure - oversized input (413, the cost ceiling)
A deliberately enormous input. `app/budget.py` prices the request before the model call and refuses it. **Nothing is spent.**

In [ ]:
huge = 'Describe your architecture. ' * 45000   # 28 x 45,000 = 1,260,000 characters
r = requests.post(f'{BASE}/ask', json={'question': huge})
print('Status:', r.status_code, '(expected 413)')
print(json.dumps(r.json(), indent=2))

---
## 14 - Memory - the audit log is scoped per user
Ask as `alice`, then read her audit log - each line records the tools the agent called. Another user sees nothing of hers.

In [ ]:
requests.post(f'{BASE}/ask', json={'question':'What is the cost ceiling?','user_id':'alice'})
print('alice:', requests.get(f'{BASE}/audit/alice', headers=ADMIN).json())
print('bob  :', requests.get(f'{BASE}/audit/bob', headers=ADMIN).json())

---
## 15 - Memory - remember, recall, and the three boundary probes
Memory is keyed by user, then session. A different user, or the same user in another session, gets `found: false` - the same answer as a key that never existed, so nothing leaks. A delete is a hard delete.

In [ ]:
pref = {'user_id': 'alice', 'session_id': 's1', 'key': 'pref'}
print('write          :', requests.post(f'{BASE}/memory', json={**pref, 'value': 'dark mode'}, headers=ADMIN).json())
print('alice / s1     :', requests.get(f'{BASE}/memory', params=pref, headers=ADMIN).json()['value'])
print('bob   / s1     :', requests.get(f'{BASE}/memory', params={**pref, 'user_id': 'bob'}, headers=ADMIN).json()['found'])
print('alice / s2     :', requests.get(f'{BASE}/memory', params={**pref, 'session_id': 's2'}, headers=ADMIN).json()['found'])
print('delete         :', requests.delete(f'{BASE}/memory', json=pref, headers=ADMIN).json())
print('after delete   :', requests.get(f'{BASE}/memory', params=pref, headers=ADMIN).json()['found'])

---
## 16 - Public-deploy guards - what a shared URL needs
A link on your resume means strangers spend **your** key. Four guards in `app/guard.py`: the admin token on `[admin]` routes (required on Render - the app will not boot there without it; the owner pastes it on the `/admin` page), a per-client limit on POSTs (`RATE_LIMIT_PER_MINUTE`, default 30), a daily model-spend budget across every caller and every call of every loop (`DAILY_BUDGET_USD`, default $1.00), and a 2 MiB body limit (413 before the body is read). The cell below trips the rate limit on purpose with a free POST (an A2A `ListTasks`) - run it last, or wait a minute before the next POST.

In [ ]:
cheap = {'jsonrpc': '2.0', 'id': 1, 'method': 'ListTasks'}
for n in range(1, 61):
    r = requests.post(f'{BASE}/a2a', json=cheap, headers={'A2A-Version': '1.0'})
    if r.status_code == 429:
        print(f'POST #{n} in this burst -> 429', r.json()['detail'], '| Retry-After:', r.headers['Retry-After'])
        break
else:
    print('no 429 - is RATE_LIMIT_PER_MINUTE set to 0?')

---
## 17 - OpenAPI / Swagger Docs
FastAPI auto-generates interactive docs.

In [ ]:
from IPython.display import display, HTML
display(HTML('<a href="http://localhost:8000/docs" target="_blank">Open Swagger UI: http://localhost:8000/docs</a>'))